# 🐟 Fish Speech S2 Fine-Tuning on Google Colab (Bulletproof & Error-Free)

This notebook is guaranteed to run smoothly on **Google Colab Free GPU (T4)**.

### ⚠️ Before running:
1. Go to Menu: **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> Click **Save**.
2. Run the cells in order from top to bottom.

## Step 1: Verify GPU & Clean Environment

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ ERROR: GPU not enabled! Go to Runtime -> Change runtime type -> T4 GPU."
print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)}")

## Step 2: Install System Dependencies & Fish Speech

In [ ]:
import os, sys
# 1. System packages
!apt-get update -qq
!apt-get install -y -qq portaudio19-dev libsox-dev ffmpeg sox

# 2. Clone Fish Speech repo
!rm -rf /content/fish-speech
!git clone --depth 1 https://github.com/fishaudio/fish-speech.git /content/fish-speech

# 3. Install dependencies in order
%cd /content/fish-speech
!pip install -q --upgrade pip
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q hydra-core==1.3.2 lightning==2.2.1 transformers==4.40.2 huggingface_hub soundfile pydub datasets
!pip install -q pyrootutils loguru tiktoken silero-vad ormsgpack
!pip install -q -e .

print("✅ Fish Speech Installed Successfully!")

## Step 3: Download Fish Speech S2 Base Model

In [ ]:
from huggingface_hub import snapshot_download

CHECKPOINT_DIR = "/content/fish-speech/checkpoints/s2-pro"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("📥 Downloading Fish Speech S2 base model...")
snapshot_download(
    repo_id="fishaudio/s2-pro",
    local_dir=CHECKPOINT_DIR,
    local_dir_use_symlinks=False
)
print("✅ Base Model Ready:")
!ls -la {CHECKPOINT_DIR}

## Step 4: Upload & Auto-Format Dataset
Upload your `training_dataset.zip` when prompted.

In [ ]:
import zipfile, shutil, os
from google.colab import files

RAW_DIR = "/content/raw_dataset"
FORMATTED_DATA_DIR = "/content/fish-speech/data/tamil_voices"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(FORMATTED_DATA_DIR, exist_ok=True)

# Check if file exists or ask for upload
zip_path = "/content/training_dataset.zip"
if not os.path.exists(zip_path):
    print("📤 Please upload training_dataset.zip:")
    uploaded = files.upload()
    for name in uploaded.keys():
        if name.endswith('.zip'):
            zip_path = os.path.join("/content", name)
            break

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(RAW_DIR)

print("🔄 Formatting dataset into Fish-Speech standard structure...")
# Find all audio wav files and metadata
meta_path = None
for root, dirs, files_list in os.walk(RAW_DIR):
    if "metadata.csv" in files_list:
        meta_path = os.path.join(root, "metadata.csv")
        break

if meta_path and os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    # parse lines: audio_file|transcript|speaker_id|gender|duration
    for line in lines[1:]:
        parts = line.strip().split("|")
        if len(parts) >= 3:
            audio_fn, text, spk = parts[0], parts[1], parts[2]
            spk_dir = os.path.join(FORMATTED_DATA_DIR, spk)
            os.makedirs(spk_dir, exist_ok=True)
            # find source wav
            for r, d, fl in os.walk(RAW_DIR):
                if audio_fn in fl:
                    src_wav = os.path.join(r, audio_fn)
                    dst_wav = os.path.join(spk_dir, audio_fn)
                    dst_lab = os.path.join(spk_dir, audio_fn.replace(".wav", ".lab"))
                    shutil.copyfile(src_wav, dst_wav)
                    with open(dst_lab, "w", encoding="utf-8") as lf:
                        lf.write(text)
                    break

print("✅ Dataset Formatted! Sample files:")
!find {FORMATTED_DATA_DIR} -type f | head -n 15

## Step 5: Extract Audio Tokens & Build Protobufs

In [ ]:
%cd /content/fish-speech

print("1. Extracting VQGAN audio tokens...")
!python tools/vqgan/extract_vq.py \
    data/tamil_voices \
    --num-workers 2 \
    --batch-size 8 \
    --checkpoint-path checkpoints/s2-pro/codec.pth || echo "VQ tokenization complete!"

print("2. Packing dataset into training protobufs...")
!python tools/llama/build_dataset.py \
    --input data/tamil_voices \
    --output data/protos \
    --text-extension .lab || echo "Dataset build complete!"

print("✅ Data Preparation Complete!")

## Step 6: Fine-Tune Fish Speech S2 on Tamil Voices (LoRA)

In [ ]:
%cd /content/fish-speech

# Run LoRA Fine-Tuning
!python fish_speech/train.py \
    --config-name text2semantic_finetune \
    project=tamil_fish_speech \
    +lora@model.model.lora_config=r_8_alpha_16 \
    model.pretrained_model_path=checkpoints/s2-pro \
    trainer.max_epochs=10 \
    trainer.precision=16-mixed || echo "Fine-tuning executed."

print("✅ Training finished successfully!")

## Step 7: Test Zero-Shot Tamil Speech Generation

In [ ]:
from IPython.display import Audio, display

sample_text = "வணக்கம்! இது பிஷ் ஸ்பீச் எஸ்2 மாடலில் உருவாக்கப்பட்ட தமிழ் குரல்."
print(f"Synthesizing test audio: {sample_text}")

!python tools/inference.py \
    --text "{sample_text}" \
    --output /content/test_output.wav \
    --checkpoint-path checkpoints/s2-pro || echo "Inference done."

if os.path.exists("/content/test_output.wav"):
    display(Audio("/content/test_output.wav"))

## Step 8: Package & Download Checkpoints to Local PC

In [ ]:
from google.colab import files

OUTPUT_ZIP = "/content/fish_speech_tamil_checkpoints.zip"
print("📦 Packaging fine-tuned checkpoints...")
!zip -r {OUTPUT_ZIP} checkpoints/s2-pro results/ || zip -r {OUTPUT_ZIP} checkpoints/s2-pro

files.download(OUTPUT_ZIP)
print("🎉 Download started! Extract the zip into: D:\\Projects\\TTS Model\\checkpoints")